<a href="https://colab.research.google.com/github/aims-ai-research-foundations/pilot-workshop/blob/main/assignments/day4/day4-course5-student.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# A LoRA Update in 60 Minutes

Course 5 — Fine-tuning & LLM Alignment.
Trace one Hausa-English example through one full LoRA update by hand, then bump the rank from 1 to 2.

NumPy only. No real model. No GPU. CPU on Colab Free runs the whole notebook in under ten seconds — the time you spend is on the predict-run-explain protocol, not on waiting for compute.

## The protocol — for every code cell

**PREDICT.** Before you run the cell, write down what you think will happen. Be specific about shapes and numbers.

**RUN.** Execute the cell. Compare the output to your prediction.

**EXPLAIN.** If your prediction was wrong, write down why. What did you misunderstand?

Skipping the predict step undercuts the whole point. The arithmetic is easy. The mental model is the hard part.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
np.set_printoptions(precision=3, suppress=True)


---
## §1.  Setup

We're adapting one linear layer **W ∈ R^{4×8}** inside a larger model. Think of it as the projection from a 4-dim embedding to 8-dim class logits — but we don't need to know what the 8 classes mean. LoRA works on any linear layer.

Our one Hausa-English example is **"abinci dadi sosai"** (food is very tasty). Its 4-dim embedding is:

| dim | 0 | 1 | 2 | 3 |
|-----|---|---|---|---|
| meaning | food | tasty | negative | intensifier |
| value   | 1 | 1 | 0 | 1 |

Target **y** is the one-hot for class 0 — "strongly positive."

We start with frozen **W = 0** (the simplest possible pre-trained state). The model currently predicts ŷ = 0 — not wrong in argmax sense, but the magnitude is way off.

### PREDICT

Before running the next cell, write down (mentally or on the worksheet):

1. The shape of `W`, `x`, `y`, and `y_hat`.
2. The value of `y_hat` when `W` is all zeros and there's no LoRA yet.

In [ ]:
# Toy setup — locked values, match the worksheet exactly.
W = np.zeros((4, 8))               # frozen layer:   4 × 8 = 32 params, all zero
x = np.array([1.0, 1.0, 0.0, 1.0])  # embedding of "abinci dadi sosai"
y = np.array([1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0])  # one-hot target, class 0

print("W shape:", W.shape, "  W sum:", W.sum())
print("x:", x)
print("y:", y)

# Current model output, no LoRA:  y_hat = x @ W
y_hat = x @ W
print("y_hat (no LoRA):", y_hat)


### EXPLAIN

`y_hat` is all zeros. With `W = 0`, the model is at "neutral" — no preference for any class. The MSE loss is `0.5 * ||y_hat - y||² = 0.5 * (1)² = 0.5`.

That's our baseline. Now we'll add a LoRA update on top of the frozen `W`.

In [ ]:
# Baseline loss: how far is y_hat from y, before any LoRA?
loss_baseline = 0.5 * np.sum((y_hat - y) ** 2)
print("baseline loss (no LoRA):", loss_baseline)


---
## §2.  Full fine-tuning baseline

Before we get to LoRA, let's see what happens if we just train all 32 entries of `W` directly. This is the "full fine-tuning" we want to avoid.

For our one example:
- Forward: `y_hat = x @ W`  (shape 8)
- Loss: `L = 0.5 * ||y_hat - y||²`
- Gradient: `∂L/∂W = x.T @ (y_hat - y)` — outer product, shape **4×8**, i.e. all 32 entries change.

### PREDICT

How many entries of `W` will be non-zero after one SGD step at η=0.1?

In [ ]:
# Full FT: compute the gradient of L w.r.t. every entry of W.
residual = y_hat - y                     # shape (8,)
grad_W   = np.outer(x, residual)         # shape (4, 8)
print("residual:", residual)
print("\ngrad_W (shape", grad_W.shape, "):")
print(grad_W)
print("\nnumber of non-zero entries in grad_W:", np.count_nonzero(grad_W))


### EXPLAIN

`grad_W` has 24 non-zero entries (every row where `x` is non-zero, every column where `residual` is non-zero). After SGD, those 24 entries of `W` will move.

That's the cost of full FT: many params change for one example. LoRA cuts that down.

In [ ]:
# One SGD step on the full W
eta = 0.1
W_full_ft = W - eta * grad_W
y_hat_full_ft = x @ W_full_ft
loss_full_ft  = 0.5 * np.sum((y_hat_full_ft - y) ** 2)
print("loss after full FT (one step):", loss_full_ft, "  (was 0.5)")
print("number of params moved:", np.count_nonzero(W - W_full_ft))


---
## §3.  LoRA factorisation

Now the LoRA trick. We freeze `W` at its pre-trained value and learn a low-rank update **ΔW = α · A · B** on top of it.

- `A ∈ R^{4×r}` — "down-projection"
- `B ∈ R^{r×8}` — "up-projection"
- `α` is a scalar that controls the magnitude of the update.

Start at rank `r = 1`. We pick:
- `A = (1, 2, -1, 0)^T`   ← shape (4, 1), 4 trainable params
- `B = (1, -1, 0, 0, 0, 0, 0, 0)`  ← shape (1, 8), 8 trainable params
- `α = 1`

Total trainable params: **12** (vs 32 for full FT).

### PREDICT

1. What's the rank of `ΔW = α · A @ B`?
2. The non-zero cells of `ΔW` lie in which rows? Which columns?
3. What's `ΔW[0, 0]`?

In [ ]:
# LoRA pieces — rank 1
alpha = 1.0
A = np.array([[1.0], [2.0], [-1.0], [0.0]])           # (4, 1)
B = np.array([[1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])  # (1, 8)
print("A shape:", A.shape, "   B shape:", B.shape)
print("trainable params:", A.size + B.size)

# Compute the low-rank update
dW = alpha * (A @ B)                                   # (4, 8)
print("\nΔW (shape", dW.shape, "):")
print(dW)


### EXPLAIN

`ΔW[i, j] = α · A[i] · B[j]`. Every row of `ΔW` is `A[i]` times the row `B`. That's exactly rank 1.

The non-zero cells lie in rows where `A` is non-zero (rows 0, 1, 2) and columns where `B` is non-zero (columns 0 and 1). 3 rows × 2 columns = 6 non-zero cells out of 32.

In [ ]:
# Verify rank-1
print("rank(ΔW):", np.linalg.matrix_rank(dW, tol=1e-9))
print("non-zero cells:", np.count_nonzero(dW))

# Confirm every row of ΔW is a scalar multiple of B
for i in range(4):
    row = dW[i]
    print(f"row {i}: A[{i}]·α = {A[i, 0] * alpha:5.2f}    row = {row}")


In [ ]:
# Now the LoRA-augmented forward pass
y_hat_lora = x @ (W + dW)
loss_lora  = 0.5 * np.sum((y_hat_lora - y) ** 2)
print("y_hat (with LoRA, before any SGD step):", y_hat_lora)
print("loss (with LoRA, before SGD):", loss_lora, "  (baseline was 0.5)")


---
## §4.  LoRA gradient

For one example, the gradient of `L = 0.5 · ||y_hat - y||²` with `y_hat = x @ (W + αAB)` factorises through `A` and `B`:

```
∂L/∂A  =  α · ((y_hat - y) · B) · x        ← shape (4, r)
∂L/∂B  =  α · (A · x) · (y_hat - y)        ← shape (r, 8)
```

Each gradient is a **scalar times a vector**. No full 4×8 matrix is ever materialised.

### PREDICT

1. The scalar `(y_hat - y) · B`. What's its value?
2. The scalar `A · x`. What's its value?
3. Will `|∇A|` or `|∇B|` be larger?

In [ ]:
# Compute the two scalars (using .item() to pull a Python float from a 1-element array)
residual_lora = y_hat_lora - y                        # (8,)
scalar_for_dA = (residual_lora @ B.T).item()          # (y_hat - y) · B
scalar_for_dB = (x @ A).item()                        # A · x
print("(y_hat - y) · B   =", scalar_for_dA)
print("A · x             =", scalar_for_dB)


In [ ]:
# Now the gradients
grad_A = alpha * scalar_for_dA * x.reshape(4, 1)      # (4, 1)
grad_B = alpha * scalar_for_dB * residual_lora.reshape(1, 8)  # (1, 8)
print("∇A (shape", grad_A.shape, "):\n", grad_A.ravel())
print("\n∇B (shape", grad_B.shape, "):\n", grad_B.ravel())
print("\n|∇A| =", np.linalg.norm(grad_A))
print("|∇B| =", np.linalg.norm(grad_B))


### EXPLAIN

The gradient flows through one direction. `∇A` is `x` scaled by 5; `∇B` is the residual scaled by 3. The vectors `∇A` and `∇B` point in different spaces — `∇A` lives in the input direction, `∇B` lives in the residual direction.

This is the central LoRA insight: the gradient stays low-dimensional. No `4×8` matrix is ever in flight.

---
## §5.  One SGD step

Step `A` and `B` at learning rate η = 0.1. Recompute `ΔW`, `y_hat`, and the loss.

### PREDICT

1. Will the new `ΔW` still be rank-1?
2. Will the loss drop below 6.5?
3. By roughly how much?

In [ ]:
# One SGD step at eta = 0.1
A_new = A - eta * grad_A
B_new = B - eta * grad_B
dW_new = alpha * (A_new @ B_new)
y_hat_new = x @ (W + dW_new)
loss_new  = 0.5 * np.sum((y_hat_new - y) ** 2)

print("A_new:", A_new.ravel())
print("B_new:", B_new.ravel())
print("\nrank(ΔW_new):", np.linalg.matrix_rank(dW_new, tol=1e-9))
print("y_hat_new:", y_hat_new)
print("loss_new :", loss_new, "  (was 6.5)")


In [ ]:
# Visualise: loss before vs after the LoRA step
labels = ["baseline\n(W=0, no LoRA)", "LoRA r=1\n(before step)", "LoRA r=1\n(after step)"]
values = [loss_baseline, loss_lora, loss_new]
fig, ax = plt.subplots(figsize=(6, 3.5))
ax.bar(labels, values, color=["#7B5BA8", "#2A1340", "#FDD633"])
for i, v in enumerate(values):
    ax.text(i, v + 0.15, f"{v:.3f}", ha="center", fontsize=10)
ax.set_ylabel("loss  L = 0.5 · ||ŷ − y||²")
ax.set_title("Loss before / during / after one LoRA SGD step")
ax.set_ylim(0, max(values) * 1.2)
plt.tight_layout()
plt.show()


---
## §6.  Bump the rank: r = 1 → r = 2

Same frozen `W`. Same `x`, `y`, `α`. Only the rank changes.

We use the **standard LoRA initialisation** for the new dimension: the new column of `A` is small-random, the new row of `B` is **zero**. That way the second outer product contributes `0` to `ΔW₂` initially — adding the rank-2 capacity doesn't change the starting prediction.

```
ΔW₂ = α · ( A[:, 0] ⊗ B[0, :]  +  A[:, 1] ⊗ B[1, :] )
```

Parameters at r=2:  **A: 4×2 = 8** plus **B: 2×8 = 16**, total **24**.
Still less than the 32 of full FT, but the model has two directions of update now.

### PREDICT

1. What's the rank of `ΔW₂` *before* the SGD step? (Hint: think about what `B[1, :] = 0` does to the outer product.)
2. What's the rank of `ΔW₂` *after* the SGD step?
3. Will the loss drop further than 0.091 (the r=1 result)?  This one is less obvious than it looks.

In [ ]:
# r = 2 with standard LoRA init: new B row starts at zero, so ΔW₂ initial = ΔW r=1
A2 = np.array([[1.0, 1.0],     # col 0 unchanged from r=1; col 1 small random (set to a chosen value here)
               [2.0, 0.0],
               [-1.0, 0.0],
               [0.0, 0.0]])    # (4, 2): col 1 = (1, 0, 0, 0)
B2 = np.array([[1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
               [0.0,  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])  # row 1 = zeros (standard LoRA init)
print("A2:", A2.shape, "B2:", B2.shape, "trainable params:", A2.size + B2.size)

dW2 = alpha * (A2 @ B2)
print("\nΔW₂ initial (rank should be 1 because B-row-1 = 0):\n", dW2)
print("\nrank(ΔW₂) initial:", np.linalg.matrix_rank(dW2, tol=1e-9))


In [ ]:
# One SGD step at r=2
y_hat_r2 = x @ (W + dW2)
residual_r2 = y_hat_r2 - y
scalar_dA2 = residual_r2 @ B2.T                # shape (2,)
scalar_dB2 = x @ A2                             # shape (2,)
grad_A2 = alpha * x.reshape(4, 1) * scalar_dA2.reshape(1, 2)
grad_B2 = alpha * scalar_dB2.reshape(2, 1) * residual_r2.reshape(1, 8)

A2_new = A2 - eta * grad_A2
B2_new = B2 - eta * grad_B2
dW2_new = alpha * (A2_new @ B2_new)
y_hat_r2_new = x @ (W + dW2_new)
loss_r2_new = 0.5 * np.sum((y_hat_r2_new - y) ** 2)

print("y_hat at r=2 before step:", y_hat_r2)
print("loss   at r=2 before step:", 0.5 * np.sum(residual_r2 ** 2))
print("\ny_hat at r=2 after  step:", y_hat_r2_new)
print("loss   at r=2 after  step:", loss_r2_new)
print("\nrank(ΔW₂_new):", np.linalg.matrix_rank(dW2_new, tol=1e-9))


### EXPLAIN

If you're surprised that the r=2 loss isn't lower than r=1, you're paying attention. This is a real LoRA fact:

- **Rank is capacity, not improvement.** Higher rank gives the model more directions of update to pick from. But with a **single** SGD step from a **specific** init, you don't always use that capacity for a win.
- **Where r=2 actually wins:** (a) over many steps — r=2 converges to a lower minimum on harder problems; (b) on multi-example datasets where one direction can't fit everything; (c) on layers where the optimal update is intrinsically high-rank.
- For our single-example single-step demo, rank 1 was already enough to hit the residual squarely. The second direction had nowhere good to step in one move.

Try this in the next cell: run 10 SGD steps each at r=1 and r=2 and watch the convergence curves.

In [ ]:
# Multi-step convergence: 10 SGD steps each at r=1 and r=2
def lora_step(A_, B_, x_, y_, alpha_, eta_):
    y_hat_ = x_ @ (alpha_ * (A_ @ B_))
    res    = y_hat_ - y_
    scalar_dA = res @ B_.T
    scalar_dB = x_ @ A_
    gA = alpha_ * x_.reshape(-1, 1) * scalar_dA.reshape(1, -1)
    gB = alpha_ * scalar_dB.reshape(-1, 1) * res.reshape(1, -1)
    return A_ - eta_ * gA, B_ - eta_ * gB, 0.5 * np.sum(res ** 2)

# r = 1 trajectory
A_r1 = np.array([[1.0], [2.0], [-1.0], [0.0]])
B_r1 = np.array([[1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
losses_r1 = []
for _ in range(10):
    A_r1, B_r1, L = lora_step(A_r1, B_r1, x, y, alpha, eta)
    losses_r1.append(L)

# r = 2 trajectory (standard LoRA init: col 1 nonzero, row 1 zero)
A_r2 = np.array([[1.0, 1.0], [2.0, 0.0], [-1.0, 0.0], [0.0, 0.0]])
B_r2 = np.array([[1.0, -1.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0],
                 [0.0,  0.0, 0.0, 0.0, 0.0, 0.0, 0.0, 0.0]])
losses_r2 = []
for _ in range(10):
    A_r2, B_r2, L = lora_step(A_r2, B_r2, x, y, alpha, eta)
    losses_r2.append(L)

print("r=1 losses over 10 steps:", [f"{L:.4f}" for L in losses_r1])
print("r=2 losses over 10 steps:", [f"{L:.4f}" for L in losses_r2])


In [ ]:
# Convergence plot — clean comparison
fig, ax = plt.subplots(figsize=(7, 4))
ax.plot(range(1, 11), losses_r1, marker="o", color="#FDD633", linewidth=2, label="r = 1  (12 params)")
ax.plot(range(1, 11), losses_r2, marker="s", color="#2A1340", linewidth=2, label="r = 2  (24 params)")
ax.set_xlabel("SGD step")
ax.set_ylabel("loss  L = 0.5 · ||ŷ − y||²")
ax.set_title("LoRA convergence over 10 SGD steps  (single example)")
ax.set_yscale("log")
ax.grid(True, which="both", alpha=0.3)
ax.legend()
plt.tight_layout()
plt.show()


---
## §7.  Reflection

You've just computed one LoRA update by hand at rank 1 and rank 2, with a parameter count of 12 → 24 (versus 32 for full FT on the same layer).

### Param-count exercise

A Gemma 7B attention layer has weight matrices of shape **4096 × 4096** (≈ 16.8M params each).

| rank | trainable params | as % of full FT |
|------|------------------|------------------|
|   1  |  4096 + 4096 = 8 192        |  0.05 %   |
|   8  |  8 · 4096 + 8 · 4096 = 65 536  |  0.39 %   |
|  64  |  64 · 4096 + 64 · 4096 = 524 288 |  3.13 %   |
| 256  | 256 · 4096 + 256 · 4096 = 2 097 152 | 12.5 % |

LoRA stops being parameter-efficient when `r · (in + out)` approaches `in · out`. For a 4096 × 4096 layer, that's when `r > 2048` — i.e., never in practice.

### Closing question

In two or three sentences:

> If you LoRA-tuned Gemma 7B at rank 8 to describe **Adire textile patterns** better — using a small dataset of, say, 200 Hausa-English image-text pairs — would you use the same `α` we used here (= 1), or scale it differently? Why?

(Hint: think about what `α` controls — magnitude of the update. In practice it's set to 8, 16, or 32 to compensate for the small rank.)

### Where to go next

- Read Hu et al., *LoRA: Low-Rank Adaptation of Large Language Models* (2021).
- Try this notebook with a real frozen `W` (random instead of zeros) — does the loss drop look the same?
- Add a second example: how does the gradient change when you accumulate over a batch?
- Stack LoRA + DPO: what happens to alignment when you train preferences on top of a LoRA-adapted SFT model?

You just computed the gradient that PEFT libraries compute under the hood. The library will be a thin wrapper around what you just did.